# Simulation Implementation Notebook (Part-by-Part)

This notebook implements the plan in incremental parts.

## Planned Parts
1. **Part 1 (implemented now):** Data inventory and schema audit for Twitter + OpenAssistant
2. Part 2: Action labeling pipeline (LLM + QA sample)
3. Part 3: Persona feature table + GMM clustering
4. Part 4: RAG index build (turn-level + conversation-level)
5. Part 5: Simulator interface (`reset`, `step`) with state extraction
6. Part 6: Bandit baselines + PPO warm-start scaffold

This run focuses only on **Part 1**.

In [2]:
from pathlib import Path
import csv
import json
import pandas as pd

# Input dataset roots
ROOT = Path(r"B:\\College\\RL\\AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service")
TWITTER_ROOT = ROOT / "twitter"
OPENASSIST_ROOT = ROOT / "OpenAssistant Conversations Dataset"

# Output folder for Part 1 artifacts
OUT_DIR = ROOT / "Simulation" / "artifacts" / "part1_data_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TWITTER_ROOT, OPENASSIST_ROOT, OUT_DIR

(WindowsPath('B:/College/RL/AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service/twitter'),
 WindowsPath('B:/College/RL/AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service/OpenAssistant Conversations Dataset'),
 WindowsPath('B:/College/RL/AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service/Simulation/artifacts/part1_data_audit'))

In [3]:
def discover_csv_files(*roots):
    files = []
    for root in roots:
        if root.exists():
            files.extend(sorted(root.rglob("*.csv")))
    return files


def count_data_rows(csv_path: Path):
    # Fast line count minus header row; robust fallback if file is empty.
    total_lines = 0
    with csv_path.open("r", encoding="utf-8", errors="ignore") as f:
        for _ in f:
            total_lines += 1
    return max(total_lines - 1, 0)


def read_header(csv_path: Path):
    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        reader = csv.reader(f)
        try:
            return next(reader)
        except StopIteration:
            return []


def sample_missingness(csv_path: Path, sample_rows=50000):
    sample_df = pd.read_csv(csv_path, nrows=sample_rows, low_memory=False)
    miss = (sample_df.isna().mean() * 100).round(2)
    out = pd.DataFrame({
        "column": miss.index,
        "missing_pct_sample": miss.values,
        "sample_rows": len(sample_df)
    })
    return out

In [4]:
csv_files = discover_csv_files(TWITTER_ROOT, OPENASSIST_ROOT)
print(f"Discovered CSV files: {len(csv_files)}")

inventory_rows = []
schema_rows = []

for p in csv_files:
    dataset_group = "twitter" if str(TWITTER_ROOT) in str(p) else "openassistant"
    headers = read_header(p)
    n_rows = count_data_rows(p)
    size_mb = round(p.stat().st_size / (1024 * 1024), 2)

    inventory_rows.append({
        "dataset_group": dataset_group,
        "file_path": str(p),
        "file_name": p.name,
        "size_mb": size_mb,
        "n_rows": n_rows,
        "n_columns": len(headers)
    })

    for col_idx, col in enumerate(headers):
        schema_rows.append({
            "dataset_group": dataset_group,
            "file_name": p.name,
            "column_index": col_idx,
            "column_name": col
        })

inventory_df = pd.DataFrame(inventory_rows).sort_values(["dataset_group", "file_name"])
schema_df = pd.DataFrame(schema_rows).sort_values(["dataset_group", "file_name", "column_index"])

display(inventory_df)
display(schema_df.head(40))

inventory_df.to_csv(OUT_DIR / "part1_file_inventory.csv", index=False)
schema_df.to_csv(OUT_DIR / "part1_schema_columns.csv", index=False)

print("Saved: part1_file_inventory.csv")
print("Saved: part1_schema_columns.csv")

Discovered CSV files: 4


,dataset_group,file_path,file_name,size_mb,n_rows,n_columns
2,openassistant,B:\College\RL\AdaptiveBandit--Contextual-Bandi...,oasst1-train.csv,119.77,771472,18
3,openassistant,B:\College\RL\AdaptiveBandit--Contextual-Bandi...,oasst1-val.csv,6.27,41102,18
0,twitter,B:\College\RL\AdaptiveBandit--Contextual-Bandi...,sample.csv,0.02,99,7
1,twitter,B:\College\RL\AdaptiveBandit--Contextual-Bandi...,twcs.csv,492.58,3003124,7


,dataset_group,file_name,column_index,column_name
14,openassistant,oasst1-train.csv,0,message_id
15,openassistant,oasst1-train.csv,1,parent_id
16,openassistant,oasst1-train.csv,2,user_id
17,openassistant,oasst1-train.csv,3,created_date
18,openassistant,oasst1-train.csv,4,text
19,openassistant,oasst1-train.csv,5,role
20,openassistant,oasst1-train.csv,6,lang
21,openassistant,oasst1-train.csv,7,review_count
22,openassistant,oasst1-train.csv,8,review_result
23,openassistant,oasst1-train.csv,9,deleted


Saved: part1_file_inventory.csv
Saved: part1_schema_columns.csv


In [5]:
missingness_frames = []

for p in csv_files:
    try:
        miss_df = sample_missingness(p, sample_rows=50000)
        miss_df.insert(0, "file_name", p.name)
        miss_df.insert(0, "dataset_group", "twitter" if str(TWITTER_ROOT) in str(p) else "openassistant")
        missingness_frames.append(miss_df)
    except Exception as e:
        missingness_frames.append(pd.DataFrame([{
            "dataset_group": "twitter" if str(TWITTER_ROOT) in str(p) else "openassistant",
            "file_name": p.name,
            "column": "__ERROR__",
            "missing_pct_sample": None,
            "sample_rows": 0,
            "error": str(e)
        }]))

missingness_df = pd.concat(missingness_frames, ignore_index=True)
missingness_df.to_csv(OUT_DIR / "part1_missingness_sample.csv", index=False)

summary = (
    inventory_df.groupby("dataset_group")[["n_rows", "n_columns", "size_mb"]]
    .agg({"n_rows": "sum", "n_columns": "mean", "size_mb": "sum"})
    .rename(columns={"n_rows": "total_rows", "n_columns": "avg_columns_per_file", "size_mb": "total_size_mb"})
    .reset_index()
)
summary.to_csv(OUT_DIR / "part1_dataset_summary.csv", index=False)

report_lines = [
    "# Part 1 Data Audit Report",
    "",
    "## Scope",
    "- Twitter root: " + str(TWITTER_ROOT),
    "- OpenAssistant root: " + str(OPENASSIST_ROOT),
    "",
    "## Outputs",
    "- part1_file_inventory.csv",
    "- part1_schema_columns.csv",
    "- part1_missingness_sample.csv",
    "- part1_dataset_summary.csv",
    "",
    "## Quick Summary",
]

for _, r in summary.iterrows():
    report_lines.append(
        f"- {r['dataset_group']}: rows={int(r['total_rows']):,}, avg_columns_per_file={r['avg_columns_per_file']:.2f}, size_mb={r['total_size_mb']:.2f}"
    )

(OUT_DIR / "part1_data_audit_report.md").write_text("\n".join(report_lines), encoding="utf-8")

display(summary)
display(missingness_df.head(40))
print("Saved: part1_missingness_sample.csv")
print("Saved: part1_dataset_summary.csv")
print("Saved: part1_data_audit_report.md")

,dataset_group,total_rows,avg_columns_per_file,total_size_mb
0,openassistant,812574,18.0,126.04
1,twitter,3003223,7.0,492.60


,dataset_group,file_name,column,missing_pct_sample,sample_rows
0,twitter,sample.csv,tweet_id,0.00,93
1,twitter,sample.csv,author_id,0.00,93
2,twitter,sample.csv,inbound,0.00,93
3,twitter,sample.csv,created_at,0.00,93
4,twitter,sample.csv,text,0.00,93
5,twitter,sample.csv,response_tweet_id,30.11,93
6,twitter,sample.csv,in_response_to_tweet_id,26.88,93
7,twitter,twcs.csv,tweet_id,0.00,50000
8,twitter,twcs.csv,author_id,0.00,50000
9,twitter,twcs.csv,inbound,0.00,50000


Saved: part1_missingness_sample.csv
Saved: part1_dataset_summary.csv
Saved: part1_data_audit_report.md


## Part 1 Complete

Part 1 provides the dataset inventory and schema baseline needed for Part 2 (action labeling).

Next planned implementation:
- Build an action-labeling pipeline that classifies agent turns into the 7-action space.
- Add a small annotation-ready export for inter-rater validation (kappa).